In [65]:
import mysql.connector
import pandas as pd
import numpy as np
import plotly.express as px

In [ ]:
con =mysql.connector.connect(
    host=host,
    user=user,
    password=password,
    database=database
)

In [67]:
cursor = con.cursor()

## Pytanie 1.
Czy któraś kategoria daje przewagę bogatym właścicielom chomików, którzy mogą sobie pozwolić na lepszy sprzęt?

In [68]:
zapytanie1 = """ SELECT 
    kategorie.nazwa_kategorii,
    kategorie.czy_wymaga_sprzetu,
    sportowcy.id_uczestnika,
    sportowcy.wartosc_wyposazenia,
    AVG(konkurencje_uczestnicy.wynik_uczestnika) AS sredni_wynik,
    COUNT(konkurencje_zawody.zwyciezca_konkurencji) AS liczba_wygranych
FROM sportowcy
JOIN konkurencje_uczestnicy ON sportowcy.id_uczestnika = konkurencje_uczestnicy.id_uczestnika
JOIN konkurencje ON konkurencje_uczestnicy.id_konkurencji = konkurencje.id_konkurencji
JOIN kategorie ON konkurencje.id_kategorii = kategorie.id_kategorii
JOIN konkurencje_zawody ON sportowcy.id_uczestnika = konkurencje_zawody.zwyciezca_konkurencji 
AND konkurencje.id_konkurencji = konkurencje_zawody.id_konkurencji
GROUP BY sportowcy.id_uczestnika, kategorie.id_kategorii;"""

In [69]:
cursor.execute(zapytanie1)
wyniki = cursor.fetchall()

if wyniki:
    df = pd.DataFrame(wyniki, columns=[
        'nazwa_kategorii', 'czy_wymaga_sprzetu', 'id_uczestnika', 'wartosc_wyposazenia', 
        'sredni_wynik', 'liczba_wygranych'
    ])

In [70]:
df.head()

,nazwa_kategorii,czy_wymaga_sprzetu,id_uczestnika,wartosc_wyposazenia,sredni_wynik,liczba_wygranych
0,Naturalna,0,5,2693,288.0000,1
1,Formuła Ch,1,5,2693,286.5000,4
2,Naturalna,0,7,4307,106.0000,1
3,Naturalna,0,8,1619,129.0000,1
4,Naturalna,0,11,1523,332.0000,1


In [71]:
fig = px.scatter(
    df, 
    x="wartosc_wyposazenia", 
    y="sredni_wynik",
    color="nazwa_kategorii",
    facet_col="czy_wymaga_sprzetu",
    size="liczba_wygranych", 
    hover_data=["id_uczestnika"],
    title="Analiza: wartość wyposażenia vs wyniki",
    labels={
        "wartosc_wyposazenia": "Wartość wyposażenia",
        "sredni_wynik": "Średni wynik"
    }
)

fig.show()

#### Krótki opis
Z porównania wyników zwycięzców w formule Ch oraz naturalnej wynika, iż wartość wyposażenia zawodnika nie wpływa na osiągane rezultaty.

## Pytanie 2.
Czy stosowanie dopingu/niedozwolonych substancji gwarantuje lepsze wyniki?

In [72]:
zapytanie2 = """ SELECT 
    sportowcy.id_uczestnika,
    AVG(wynik_uczestnika) AS sredni_wynik,
    COUNT(id_kontroli) AS liczba_dyskwalifikacji,
    IF(COUNT(id_kontroli) > 0, 'Zdyskwalifikowany', 'Niezdyskwalifikowany') AS status_antydopingowy
FROM sportowcy
JOIN konkurencje_uczestnicy ON sportowcy.id_uczestnika = konkurencje_uczestnicy.id_uczestnika
LEFT JOIN zdyskwalifikowani ON sportowcy.id_uczestnika = zdyskwalifikowani.id_uczestnika
GROUP BY sportowcy.id_uczestnika;
"""

In [73]:
cursor.execute(zapytanie2)
wyniki2 = cursor.fetchall()

if wyniki2:
    df_doping = pd.DataFrame(wyniki2, columns=[
        'id_uczestnika', 'sredni_wynik', 'liczba_dyskwalifikacji', 'status_antydopingowy'
    ])

In [74]:
df_doping.head()

,id_uczestnika,sredni_wynik,liczba_dyskwalifikacji,status_antydopingowy
0,1,309.0000,0,Niezdyskwalifikowany
1,2,328.0000,1,Zdyskwalifikowany
2,3,183.3333,0,Niezdyskwalifikowany
3,4,193.3333,0,Niezdyskwalifikowany
4,5,276.6667,0,Niezdyskwalifikowany


In [75]:
fig2 = px.box(
    df_doping, 
    x="status_antydopingowy", 
    y="sredni_wynik", 
    color="status_antydopingowy",
    points="all",
    title="Porównanie wyników:'Niezdyskwalifikowani' vs 'Zdyskwalifikowani'",
    labels={"status_antydopingowy": "Status kontroli", "sredni_wynik": "Średni wynik"},
    color_discrete_map={'Czysty': 'green', 'Zdyskwalifikowany': 'red'}
)

fig2.show()

#### Króki opis
Mediana w obu wynikach jest zbiżona, zatem stosowanie niedozwolonych nie gwarantuje wyższych wyników w zawodach.